<a href="https://colab.research.google.com/github/elhamod/BA305_Fall_2026/blob/main/Session%2008%20-%20k-Nearest%20Neighbors/lab4_k_Nearest_Neighbors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lab 4: k-Nearest Neighbors (kNN)**

![picture](https://drive.google.com/uc?export=view&id=1WImawQOK5kND12WwThEMzG0mN6YMRYe2)


*Personal Loan Acceptance*

Universal Bank is a relatively young bank growing rapidly in terms of overall customer acquisition. The majority of these customers are liability customers (depositors) with varying sizes of relationship with the bank. The customer base of asset customers (borrowers) is quite small, and **the bank is interested in expanding this base rapidly to bring in more loan business.** In particular, it wants to explore ways of converting its depositors to personal loan customers (while retaining them as depositors).

A campaign that the bank ran last year for depositors showed a healthy conversion rate of over 9% success. This has encouraged the retail marketing department to devise smarter campaigns with better target marketing. **The goal is to use
k-NN to predict whether a new customer will accept a loan offer.** This will serve as the basis for the design of a new campaign.

The file _UniversalBank.csv_ contains data on 5000 customers. The data include customer demographic information (age, income, etc.), the customer’s relationship with the bank (mortgage, securities account, etc.), and the customer response to the last personal loan campaign (Personal Loan). Among these 5000 customers, only 480 (= 9.6%) accepted the personal loan that was offered to them in the earlier campaign.

Columns: Age (years), Experience (years of work experience), Income (in thousands of USD), Family (number of family members), CCAvg (current credit in thousands of USD), Education (1 undergrad, 2 grad, 3 professional degrees), mortgage (0 or 1), personal loan (0 or 1), securities account (0 or 1), CD account (0 or 1), Online (0 or 1) and CreditCard (0 or 1).

## Preprocessing

In [ ]:
# install & import required packages
%matplotlib inline
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

from tqdm import tqdm # progress slider for ``for'' loops

# Set print options to suppress scientific notation
np.set_printoptions(suppress=True)

In [ ]:
# load the data
bank_df = pd.read_csv('https://raw.githubusercontent.com/AnalyticsArmory/data/main/UniversalBank.csv')
bank_df.info()

The variable we are interested in predicting is called "Personal Loan", which is a binary variable. A 0 means the customer did not accept the loan offer. A 1 means the customer accepted the loan offer.

In [ ]:
# drop ID and ZIP Code columns
bank_df = bank_df.drop(columns=['ID', 'ZIP Code'])

# make sure that the result is as expected
bank_df.head()

In [ ]:
# modify column names
bank_df.columns = [x.replace(' ', '_') for x in bank_df.columns]
list(bank_df.columns)

In [ ]:
# create dummy variables for categorical variables
# we need to transform Education (1,2,3) into a binary categorical variable
# because this is what the k-nn classifier requires. To do this:
# first, tell python that education values are not numbers,
# but rather placeholders for categories
bank_df['Education'] = bank_df['Education'].astype('category')

In [ ]:
# then use the get_dummies function to convert to categorical
# it will look for any 'category' type variable and convert it
# careful: it automatically drops the first dummy variables unless you tell it not to
bank_df = pd.get_dummies(bank_df, prefix_sep='_', drop_first=False)
bank_df.head()

In [ ]:
# Separate X (input features/aka predictors) from y (target)

# get list of column names
predictors = list(bank_df.columns)
# remove the 'Personal_Loan' column, since this is what we will try to predict
predictors.remove('Personal_Loan')

# Store predictors and target into X and y, respectively
X = bank_df[predictors]
y = bank_df['Personal_Loan']

In [ ]:
# split dataset into training (60%) and test (40%) sets
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.4, random_state=2, stratify=y)
print('Training set:', X_train.shape, 'Testing set:', X_test.shape)

# Note, the 'stratify' option ensures that both the y training and testing data
# have the same proportion of 1's and 0's. For instance, if y_train has
# 10% of entries that are 1 and 90% that are 0, then the stratify option will
# ensure that the y_test data also has 10% 1's and 90% 0's

In [ ]:
# Standardize training and validation features using 'StandatdScaler()'
# (a slightly different method than what we did in the PCA lab)

# the first line defines the scaling object
scaler = preprocessing.StandardScaler()

# the second line specifies which data to use to compute means and variances
scaler.fit(X_train)
# important: in this step, you should fit the scaler only to training data,
# and not the testing data. We assume testing data is never available to us
# in the training stage

# the third line scales the data using the means and variances computed in the
# previous step
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Note, there is no need to scale the target y variable, since this is what
# we are trying to predict

## Run the k-NN Model!

In [ ]:
# Run the k-NN model with a random guess about the neighboors, set k=1 for instance
knn = KNeighborsClassifier(n_neighbors=1)
# Specify the training features (X_train) and the outcome they lead to (y_train)
# Important: only use the training data at this step. Do not use test data.
knn.fit(X_train_scaled, y_train)

# Now that the model is done, we can use it to predict whether the new customers
# from the testing data, will accept a loan or not. For this, we feed the
# testing features X_test into the prediction function.
y_pred = knn.predict(X_test_scaled)
print('Accuracy:', accuracy_score(y_test, y_pred))

In [ ]:
# Choosing the best k for the validation set
# using a 'for' loop and range(start, stop, step)
results = []
for k in tqdm(range(1, 51, 1)):

    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train_scaled, y_train)

    # create a dictionary to store the results
    results.append({
        'k': k,
        'accuracy': accuracy_score(y_test, knn.predict(X_test_scaled))
    })

# convert results to a pandas dataframe for better visualization
results_df = pd.DataFrame(results)
results_df

In [ ]:
# plot accuracy vs. k
results_df.plot.scatter(x='k', y='accuracy', xlim=[0, 50]);

In [ ]:
# Find the max value and associated index
max_val = results_df['accuracy'].max()
max_val_idx = results_df['accuracy'].idxmax()

print("Max value =", max_val, '|', " Best k =", results_df['k'][max_val_idx])

We choose the best k, which minimizes the misclassification rate in the validation set. Our best k is `k=3`

In [ ]:
# Show the confusion matrix and accuracy for the validation data, using k = 3
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)
print(confusion_matrix(y_test, y_pred))
print('Accuracy:', accuracy_score(y_test, y_pred))

## Let's now create a new customer that the model has never seen before...

In [ ]:
# We will get the column names to help
list(bank_df.columns)

In [ ]:
# define a new customer (remove personal loan column)
newCustomer = pd.DataFrame([
{
 'Age':40,
 'Experience':10,
 'Income':84,
 'Family':2,
 'CCAvg':2,
 'Mortgage':0,
 'Securities_Account':0,
 'CD_Account':0,
 'Online':1,
 'CreditCard':1,
 'Education_1':0,
 'Education_2':1,
 'Education_3':0
 }
 ])
newCustomer

In [ ]:
# Important: In practice, since this is an entirely new customer, we can
# now use all the data to run knn (both training and testing).
# We also need to rescale using the full dataset.
scaler.fit(X)
X_scaled = scaler.transform(X)

# predict the class of the New Customer
# Don't forget to Standardize the new Customer!
newCustomerNorm = scaler.transform(newCustomer)

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_scaled, y)

pred_y = knn.predict(X_scaled)

knn.predict(newCustomerNorm)

New customer is predicted to not accept a loan offer.

In [ ]:
# Who are the 3 nearest kneighbors? and how far are they?
# you can use the kneighbors() command to find out
distances, indices = knn.kneighbors(newCustomerNorm)
print('Indices of the 3 nearest neighbors:', indices)
print('Distances of the 3 nearest neighbors:', distances)

## Bonus: to recover original customer data after re-scaling, use the inverse_transform() function as per below

In [ ]:
scaler.inverse_transform(newCustomerNorm)